In [1]:
%matplotlib inline
import pandas as pd
import numpy as np

In [2]:
def slow_hamming_distance(s1, s2):
    if len(s1) != len(s2):
        print(s1,s2)
        raise ValueError("Strand lengths are not equal!")
    term_list = []
    for ch1,ch2 in zip(s1,s2):
        if ch1 == "N" or ch2 == "N":
            term_list.append(False)
        else:
            term_list.append(ch1 != ch2)
    result = sum(term_list)
    return result

In [3]:
# Add fiducial sequences to the 
# Import nanopore data
df_nanopore = pd.read_csv("/home/lag36/scratch/lag36/2024-08-26_lLAG8_Nanopore/lLAG8_final_df.tsv", sep="\t").drop(columns=["Unnamed: 0"])

In [4]:
df_nanopore

,barcodeid,barcode,consensus,reference,alignmentstart,cigar,subsample,barcode depth,closest_match,aligned_cons,...,Closest Hamming Distance,opLAG1_id,locus_tag,offset,Gene,sgRNA Sequence,Target Sequence,Category,Predicted_Efficacy,TargetID
0,0,000000000000000001100000111010,ATTTGTCCTACTCAGGAGAGCGTTCACCGACAAACAACAGATAAAA...,ATTTGTCCTACTCAGGAGAGCGTTCACCGACAAACAACAGATAAAA...,1,1091M,NaN,227,4,ATTTGTCCTACTCAGGAGAGCGTTCACCGACAAACAACAGATAAAA...,...,4,2943,BSU_16630,250.0,infB,TTTCCGTCGCGAATTTTATT,AATAAAATTCGAGACGGAAA,essential,0.764367,964
1,1,000000000000000001110111111011,ATTTGTCCTACTCAGGAGAGCGTTCACCGACAAACAACAGATAAAA...,ATTTGTCCTACTCAGGAGAGCGTTCACCGACAAACAACAGATAAAA...,1,1091M,NaN,518,3,ATTTGTCCTACTCAGGAGAGCGTTCACCGACAAACAACAGATAAAA...,...,3,2172,BSU_15160,284.0,pbpB,TTGAGGATGCTTGACATCAG,CTGATGTCAAGCATCCTCAG,essential,1.097444,699
2,2,000000000000000011111001111101,ATTTGTCCTACTCAGGAGAGCGTTCACCGACAAACAACAGATAAAA...,ATTTGTCCTACTCAGGAGAGCGTTCACCGACAAACAACAGATAAAA...,1,682M1D15M1D236M1D155M,NaN,223,3,ATTTGTCCTACTCAGGAGAGCGTTCACCGACAAACAACAGATAAAA...,...,3,183,BSU_00380,40.0,metS,TTTCCGCTCGGATAATAAAA,ATTTATTATCCGAGCGGAAA,essential,0.154895,62
3,3,000000000000000100001011111011,ATTTGTCCTACTCAGGAGAGCGTTCACCGACAAACAACAGATAAAA...,ATTTGTCCTACTCAGGAGAGCGTTCACCGACAAACAACAGATAAAA...,1,1091M,NaN,219,4,ATTTGTCCTACTCAGGAGAGCGTTCACCGACAAACAACAGATAAAA...,...,4,3907,BSU_25290,392.0,era,ATTAGCAGAAGCAATTGGTA,GACCAATTGCTTCTGCTAAT,essential,0.063582,1284
4,4,000000000000000101111101111000,ATTTGTCCTACTCAGGAGAGCGTTCACCGACAAACAACAGATAAAA...,ATTTGTCCTACTCAGGAGAGCGTTCACCGACAAACAACAGATAAAA...,1,1091M,NaN,408,4,ATTTGTCCTACTCAGGAGAGCGTTCACCGACAAACAACAGATAAAA...,...,4,3785,BSU_24350,22.0,accB,TAATTGCTTTAATCAGCTCG,CGAGCTGATTAAAGCAATTG,essential,1.024484,1238
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
117005,123890,111111110111111110011010100111,ATTTGTCCTACTCAGGAGAGCGTTCACCGACAAACAACAGATAAAA...,ATTTGTCCTACTCAGGAGAGCGTTCACCGACAAACAACAGATAAAA...,1,925M1D165M,NaN,460,4,ATTTGTCCTACTCAGGAGAGCGTTCACCGACAAACAACAGATAAAA...,...,4,4718,BSU_29230,181.0,dnaE,AAGCCGTCAAATCGATTATT,AATAATCGGTTTGACGGCTT,essential,0.746577,1566
117006,123891,111111110111111110101110111110,ATTTGTCCTACTCAGGAGAGCGTTCACCGACAAACAACAGATAAAA...,ATTTGTCCTACTCAGGAGAGCGTTCACCGACAAACAACAGATAAAA...,1,207M1I884M,NaN,314,4,ATTTGTCCTACTCAGGAGAGCGTTCACCGACAAACAACAGATAAAA...,...,4,3412,BSU_22450,58.0,cca,CCCCGACAAAATAAACCTGA,TCAGGCTTATTTTGTCGGGG,essential,0.456055,1123
117007,123892,111111110111111111100110101001,ATTTGTCCTACTCAGGAGAGCGTTCACCGACAAACAACAGATAAAA...,ATTTGTCCTACTCAGGAGAGCGTTCACCGACAAACAACAGATAAAA...,1,533M1D557M,NaN,558,3,ATTTGTCCTACTCAGGAGAGCGTTCACCGACAAACAACAGATAAAA...,...,3,2639,BSU_16020,92.0,rimM,TCCCGGCTTGTATCGTTGTT,AAGAACGATACAAGCCGGGA,essential,0.577618,862
117008,123893,111111111100111100001101000011,ATTTGTCCTACTCAGGAGAGCGTTCACCGACAAACAACAGATAAAA...,ATTTGTCCTACTCAGGAGAGCGTTCACCGACAAACAACAGATAAAA...,1,1091M,NaN,257,4,ATTTGTCCTACTCAGGAGAGCGTTCACCGACAAACAACAGATAAAA...,...,4,6917,BSU_22660,310.0,trpC,AAAAAATCTTTTCTGAGTAC,GTACTCAGAAAAGATTTTAT,auxotroph,0.775967,2201


In [8]:
barcode_list = df_nanopore["barcode"]

In [5]:
df_nanopore.columns

Index(['barcodeid', 'barcode', 'consensus', 'reference', 'alignmentstart',
       'cigar', 'subsample', 'barcode depth', 'closest_match', 'aligned_cons',
       'split_ref', 'split_align', 'hamm_ref', 'UMI', 'sgRNA',
       'Closest Hamming Distance', 'opLAG1_id', 'locus_tag', 'offset', 'Gene',
       'sgRNA Sequence', 'Target Sequence', 'Category', 'Predicted_Efficacy',
       'TargetID'],
      dtype='object')

In [7]:
barcode_fiducial_1 = '1'*30 # LAG540
barcode_fiducial_2 = '10'*15 # LAG550
barcode_fiducial_3 = '01'*15 # LAG544
fiducial_barcodes = [barcode_fiducial_1, barcode_fiducial_2, barcode_fiducial_3]

In [8]:
# Get hamming distance matrix (for 2 barcodes)
hamming_distance_matrix = np.zeros((len(fiducial_barcodes), len(fiducial_barcodes)))
for i, barcode_1 in enumerate(fiducial_barcodes):
    for j, barcode_2 in enumerate(fiducial_barcodes):
        hamming_distance_matrix[i,j] = slow_hamming_distance(barcode_1, barcode_2)

# Place infinity in the diagonal
np.fill_diagonal(hamming_distance_matrix, np.inf)

# Get the min hamming distances across all fiducials
min_ham_distance_fiducials = np.min(hamming_distance_matrix, axis=1).astype(int)

# Get the list of closest hamming distances for each barcode
closest_hamming_distance = np.zeros(len(fiducial_barcodes), dtype=int)
for i, barcode in enumerate(fiducial_barcodes):
    print(i, barcode)
    closest_hamming_distance[i] = np.min([df_nanopore.apply(lambda x: slow_hamming_distance(x["barcode"], barcode), axis=1).min(), min_ham_distance_fiducials[i]])

# Add an entry for the fiducial sequences
df_nanopore_fiducials = (pd.concat([df_nanopore, pd.DataFrame({"barcode":fiducial_barcodes,"Closest Hamming Distance":closest_hamming_distance,
                                               "barcodeid": np.arange(len(fiducial_barcodes)) + df_nanopore['barcodeid'].max() + 1,
                                               "closest_match": closest_hamming_distance,
                                               "Category": ["fiducial"]*len(fiducial_barcodes)
                                               })])
                                               .reset_index(drop=True))


0 111111111111111111111111111111
1 101010101010101010101010101010
2 010101010101010101010101010101


In [10]:
df_nanopore_fiducials.to_csv("/home/lag36/scratch/lag36/2024-08-26_lLAG8_Nanopore/lLAG8_final_df_fiducials.tsv", sep="\t")

In [ ]:
pd.read_csv("/home/lag36/scratch/lag36/2024-08-26_lLAG8_Nanopore/lLAG8_final_df.tsv",delimiter="\t",index_col=0)

In [11]:
df = pd.read_csv("/home/lag36/scratch/lag36/2024-08-26_lLAG8_Nanopore/lLAG8_final_df_fiducials.tsv",delimiter="\t",index_col=0)

In [12]:
df

,barcodeid,barcode,consensus,reference,alignmentstart,cigar,subsample,barcode depth,closest_match,aligned_cons,...,Closest Hamming Distance,opLAG1_id,locus_tag,offset,Gene,sgRNA Sequence,Target Sequence,Category,Predicted_Efficacy,TargetID
0,0,000000000000000001100000111010,ATTTGTCCTACTCAGGAGAGCGTTCACCGACAAACAACAGATAAAA...,ATTTGTCCTACTCAGGAGAGCGTTCACCGACAAACAACAGATAAAA...,1.0,1091M,NaN,227.0,4,ATTTGTCCTACTCAGGAGAGCGTTCACCGACAAACAACAGATAAAA...,...,4,2943.0,BSU_16630,250.0,infB,TTTCCGTCGCGAATTTTATT,AATAAAATTCGAGACGGAAA,essential,0.764367,964.0
1,1,000000000000000001110111111011,ATTTGTCCTACTCAGGAGAGCGTTCACCGACAAACAACAGATAAAA...,ATTTGTCCTACTCAGGAGAGCGTTCACCGACAAACAACAGATAAAA...,1.0,1091M,NaN,518.0,3,ATTTGTCCTACTCAGGAGAGCGTTCACCGACAAACAACAGATAAAA...,...,3,2172.0,BSU_15160,284.0,pbpB,TTGAGGATGCTTGACATCAG,CTGATGTCAAGCATCCTCAG,essential,1.097444,699.0
2,2,000000000000000011111001111101,ATTTGTCCTACTCAGGAGAGCGTTCACCGACAAACAACAGATAAAA...,ATTTGTCCTACTCAGGAGAGCGTTCACCGACAAACAACAGATAAAA...,1.0,682M1D15M1D236M1D155M,NaN,223.0,3,ATTTGTCCTACTCAGGAGAGCGTTCACCGACAAACAACAGATAAAA...,...,3,183.0,BSU_00380,40.0,metS,TTTCCGCTCGGATAATAAAA,ATTTATTATCCGAGCGGAAA,essential,0.154895,62.0
3,3,000000000000000100001011111011,ATTTGTCCTACTCAGGAGAGCGTTCACCGACAAACAACAGATAAAA...,ATTTGTCCTACTCAGGAGAGCGTTCACCGACAAACAACAGATAAAA...,1.0,1091M,NaN,219.0,4,ATTTGTCCTACTCAGGAGAGCGTTCACCGACAAACAACAGATAAAA...,...,4,3907.0,BSU_25290,392.0,era,ATTAGCAGAAGCAATTGGTA,GACCAATTGCTTCTGCTAAT,essential,0.063582,1284.0
4,4,000000000000000101111101111000,ATTTGTCCTACTCAGGAGAGCGTTCACCGACAAACAACAGATAAAA...,ATTTGTCCTACTCAGGAGAGCGTTCACCGACAAACAACAGATAAAA...,1.0,1091M,NaN,408.0,4,ATTTGTCCTACTCAGGAGAGCGTTCACCGACAAACAACAGATAAAA...,...,4,3785.0,BSU_24350,22.0,accB,TAATTGCTTTAATCAGCTCG,CGAGCTGATTAAAGCAATTG,essential,1.024484,1238.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
117008,123893,111111111100111100001101000011,ATTTGTCCTACTCAGGAGAGCGTTCACCGACAAACAACAGATAAAA...,ATTTGTCCTACTCAGGAGAGCGTTCACCGACAAACAACAGATAAAA...,1.0,1091M,NaN,257.0,4,ATTTGTCCTACTCAGGAGAGCGTTCACCGACAAACAACAGATAAAA...,...,4,6917.0,BSU_22660,310.0,trpC,AAAAAATCTTTTCTGAGTAC,GTACTCAGAAAAGATTTTAT,auxotroph,0.775967,2201.0
117009,123894,111111111111010000010011101001,ATTTGTCCTACTCAGGAGAGCGTTCACCGACAAACAACAGATAAAA...,ATTTGTCCTACTCAGGAGAGCGTTCACCGACAAACAACAGATAAAA...,1.0,1091M,NaN,237.0,5,ATTTGTCCTACTCAGGAGAGCGTTCACCGACAAACAACAGATAAAA...,...,5,8084.0,NaN,NaN,NaN,CAGGCCCTTGACCCACAGTG,CACTGTGGGTCAAGGGCCTG,control,NaN,2537.0
117010,123895,111111111111111111111111111111,NaN,NaN,NaN,NaN,NaN,NaN,5,NaN,...,5,NaN,NaN,NaN,NaN,NaN,NaN,fiducial,NaN,NaN
117011,123896,101010101010101010101010101010,NaN,NaN,NaN,NaN,NaN,NaN,3,NaN,...,3,NaN,NaN,NaN,NaN,NaN,NaN,fiducial,NaN,NaN


In [ ]:
df['Closest Hamming Distance'].value_counts()/df['Closest Hamming Distance'].value_counts().sum()